In [1]:
!pip install -U peft datasets sacrebleu sentencepiece accelerate evaluate bitsandbytes transformers==4.44.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 104.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 37.8 MB/s eta 0:00:00:00:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 21.7 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: tokenizers
    Found existing installation:

In [2]:
import pandas as pd
import sacrebleu
import numpy as np
import torch
import transformers
import peft
import datasets
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments


print("torch", torch.__version__)
print("transformers", transformers.__version__, "| peft", peft.__version__)
print("datasets", datasets.__version__)
print("GPU count:", torch.cuda.device_count())

2026-04-05 03:19:14.810515: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775359154.946485     106 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775359154.986877     106 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775359155.325686     106 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775359155.325717     106 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775359155.325719     106 computation_placer.cc:177] computation placer alr

torch 2.10.0+cu128
transformers 4.44.0 | peft 0.18.1
datasets 4.8.4
GPU count: 1


In [ ]:
from huggingface_hub import login
hf_token = "your_token"
login(token=hf_token)

# Load data

In [4]:
from datasets import Dataset, DatasetDict, load_dataset
import random
import pandas as pd
import os

# ===== Load MedEV =====
ds = load_dataset("nhuvo/MedEV")
dataset = ds["train"]

offset = 340897
num_pairs = 340897

pairs = []

for i in range(num_pairs):
    en = dataset[i]["text"]
    vi = dataset[i + offset]["text"]

    pairs.append({
        "en": en,
        "vi": vi
    })

# ===== Shuffle với seed =====
random.seed(42)
random.shuffle(pairs)

# ===== Lấy 30k sample =====
pairs_30k = pairs[:30000]

# ===== Split train / val / test =====
train_size = int(0.8 * len(pairs_30k))
val_size = int(0.1 * len(pairs_30k))

train_pairs = pairs_30k[:train_size]
val_pairs   = pairs_30k[train_size:train_size+val_size]
test_pairs  = pairs_30k[train_size+val_size:]

# ===== Convert sang DataFrame =====
train_df = pd.DataFrame(train_pairs)
val_df   = pd.DataFrame(val_pairs)
test_df  = pd.DataFrame(test_pairs)

# ===== Convert sang HuggingFace Dataset =====
raw = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df),
    "test": Dataset.from_pandas(test_df)
})

print(raw)

README.md: 0.00B [00:00, ?B/s]

train.en.txt:   0%|          | 0.00/48.6M [00:00<?, ?B/s]

train.vi.txt:   0%|          | 0.00/61.9M [00:00<?, ?B/s]

val.en.new.txt: 0.00B [00:00, ?B/s]

val.vi.new.txt: 0.00B [00:00, ?B/s]

...

test.vi.new.txt: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/681794 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/17878 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/17920 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['en', 'vi'],
        num_rows: 24000
    })
    validation: Dataset({
        features: ['en', 'vi'],
        num_rows: 3000
    })
    test: Dataset({
        features: ['en', 'vi'],
        num_rows: 3000
    })
})


In [5]:
for i in range(5):
    print("EN:", pairs[i]["en"])
    print("VI:", pairs[i]["vi"])
    print("-"*60)

EN: Cystoscopy was negative.
VI: Nội soi bàng quang bình thường.
------------------------------------------------------------
EN: Objectives: The aim of this study is to determine the prevalence of acceptance of modern contraceptive methods and associated factors in postabortion women in postabortion women in reproductive health care center Long An province.
VI: Mục tiêu: Xác định tỷ lệ chấp nhận BPTT hiện đại ở phụ nữ sau phá thai tại Trung tâm chăm sóc sức khoẻ sinh sản tỉnh Long An.
------------------------------------------------------------
EN: The research sample included 210 patients. - 120 atheromatous patients; - 90 non - atheromatous patients.
VI: - 120 bệnh nhân XVĐM và 90 bệnh nhân không XVĐM.
------------------------------------------------------------
EN: Outcome of delayed nephrectomy after neoadjuvant chemotherapy in management of pediatric nephroblastoma
VI: Kết quả điều trị phẫu thuật cắt thận trì hoãn sau hoá trị trong điều trị bướu nguyên bào thận ở trẻ em
---------

# Prepare model

In [6]:
model_name = "VietAI/envit5-translation"

In [7]:
from transformers import AutoModelForSeq2SeqLM
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

config.json:   0%|          | 0.00/721 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

In [8]:
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/1.10M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


# LoRA config

In [9]:
import re
def get_num_layers(model):
    numbers = set()
    for name, _ in model.named_parameters():
        for number in re.findall(r'\d+', name):
            numbers.add(int(number))
    return max(numbers)

def get_last_layer_linears(model):
    names = []
    
    num_layers = get_num_layers(model)
    for name, module in model.named_modules():
        if str(num_layers) in name and not "encoder" in name:
            if isinstance(module, torch.nn.Linear):
                names.append(name)
    return names

In [10]:
get_num_layers(model)

11

In [11]:
get_last_layer_linears(model)

['decoder.block.11.layer.0.SelfAttention.q',
 'decoder.block.11.layer.0.SelfAttention.k',
 'decoder.block.11.layer.0.SelfAttention.v',
 'decoder.block.11.layer.0.SelfAttention.o',
 'decoder.block.11.layer.1.EncDecAttention.q',
 'decoder.block.11.layer.1.EncDecAttention.k',
 'decoder.block.11.layer.1.EncDecAttention.v',
 'decoder.block.11.layer.1.EncDecAttention.o',
 'decoder.block.11.layer.2.DenseReluDense.wi_0',
 'decoder.block.11.layer.2.DenseReluDense.wi_1',
 'decoder.block.11.layer.2.DenseReluDense.wo']

In [12]:
config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["k","q","v","o"],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)
model = get_peft_model(model, config)
model.print_trainable_parameters()

trainable params: 3,538,944 || all params: 278,641,920 || trainable%: 1.2701


In [13]:
model

PeftModelForSeq2SeqLM(
  (base_model): LoraModel(
    (model): T5ForConditionalGeneration(
      (shared): Embedding(50048, 768)
      (encoder): T5Stack(
        (embed_tokens): Embedding(50048, 768)
        (block): ModuleList(
          (0): T5Block(
            (layer): ModuleList(
              (0): T5LayerSelfAttention(
                (SelfAttention): T5Attention(
                  (q): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=False)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.05, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=768, out_features=16, bias=False)
                    )
                    (lora_B): ModuleDict(
                      (default): Linear(in_features=16, out_features=768, bias=False)
                    )
                    (lora_embedding_A): ParameterDict()
            

# Format finetune data

In [16]:
def preprocess_function(examples):
    inputs = ["en: " + doc for doc in examples["en"]]
    model_inputs = tokenizer(inputs, max_length=512, truncation=True)

    targets = ["vi: " + doc for doc in examples["vi"]]
    labels = tokenizer(text_target=targets, max_length=512, truncation=True)

    labels_input_ids = labels["input_ids"]
    labels_input_ids = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels_input_ids
    ]
    
    model_inputs["labels"] = labels_input_ids
    return model_inputs

tokenized_raw = raw.map(
    preprocess_function,
    batched=True,
    remove_columns=raw["train"].column_names
)

Map:   0%|          | 0/24000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [17]:
print(tokenized_raw["train"][0].keys())

dict_keys(['input_ids', 'attention_mask', 'labels'])


In [18]:
print(tokenized_raw["train"][0])

{'input_ids': [1055, 49804, 9, 1432, 47316, 629, 10749, 49774, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [1875, 49804, 939, 10723, 17398, 4625, 1318, 1746, 49774, 1]}


# Train

In [19]:
import evaluate
import numpy as np

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    pad_to_multiple_of=8
)
metric = evaluate.load("sacrebleu")

def postprocess(preds, labels):
    preds = [p.strip() for p in preds]
    preds = [p.replace("vi:", "").replace("vi :", "").strip() for p in preds]
    
    labels = [l.strip() for l in labels]
    labels = [[l.replace("vi:", "").replace("vi :", "").strip()] for l in labels]
    
    preds = [' '.join(p.split()) for p in preds]
    labels = [[' '.join(l[0].split())] for l in labels]
    
    return preds, labels

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    preds = np.clip(preds, 0, len(tokenizer) - 1)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    decoded_preds, decoded_labels = postprocess(decoded_preds, decoded_labels)
    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": result["score"]}

In [20]:
targs = Seq2SeqTrainingArguments(
    output_dir="./finetuned_model",
    fp16=True,       
    bf16=False,
    per_device_train_batch_size=8, 
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4, 
    learning_rate=2e-4,
    num_train_epochs=1,    
    logging_steps=250,
    save_strategy="steps",
    eval_strategy="epoch",
    save_steps=250,        
    save_total_limit=2,
    report_to="none",
    group_by_length=True, 
    ddp_find_unused_parameters=False,
    dataloader_num_workers=4,
    remove_unused_columns=False,
    predict_with_generate=True,
    generation_max_length=512
)

In [21]:
trainer = Seq2SeqTrainer(
    model=model,
    args=targs,
    train_dataset=tokenized_raw["train"],
    eval_dataset=tokenized_raw["test"], 
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [22]:
print("Start training...")
trainer.train()

Start training...


Epoch,Training Loss,Validation Loss,Bleu
1,1.108800,1.013372,46.358929


TrainOutput(global_step=750, training_loss=1.1241471354166668, metrics={'train_runtime': 1019.8357, 'train_samples_per_second': 23.533, 'train_steps_per_second': 0.735, 'total_flos': 1471572176928768.0, 'train_loss': 1.1241471354166668, 'epoch': 1.0})

In [23]:
for name, param in model.named_parameters():
    if "lora_A" in name:
        print(name, param.data.abs().mean().item())
        break

base_model.model.encoder.block.0.layer.0.SelfAttention.q.lora_A.default.weight 0.018500560894608498


# Save model

In [25]:
trainer.save_model("/kaggle/working/final_translation_model")
tokenizer.save_pretrained("/kaggle/working/final_translation_model")

('/kaggle/working/final_translation_model/tokenizer_config.json',
 '/kaggle/working/final_translation_model/special_tokens_map.json',
 '/kaggle/working/final_translation_model/spiece.model',
 '/kaggle/working/final_translation_model/added_tokens.json')

# Chuyển thành file zip để tải xuống và lưu

In [26]:
import shutil

shutil.make_archive(
    "envit5_lora_model",
    "zip",
    "/kaggle/working/final_translation_model"
)

'/kaggle/working/envit5_lora_model.zip'

In [27]:
import shutil

shutil.make_archive(
    "finetuned_model",
    "zip",
    "/kaggle/working/finetuned_model"
)

'/kaggle/working/finetuned_model.zip'